In [1]:
import pandas as pd

df = pd.read_parquet(
    "../data/processed/papers.parquet"
)

print("Shape:", df.shape)

Shape: (944, 15)


In [2]:
pd.set_option("display.max_colwidth", 150)

df[
    ["title", "year", "citation_count"]
].sort_values(
    "citation_count",
    ascending=False
).head(50)

,title,year,citation_count
424,SciPy 1.0: fundamental algorithms for scientific computing in Python,2020,39949
385,Large language models encode clinical knowledge,2023,3909
403,"Pre-train, Prompt, and Predict: A Systematic Survey of Prompting Methods in Natural Language Processing",2022,3868
388,Dense Passage Retrieval for Open-Domain Question Answering,2020,3352
159,Affordance-Compiled Intelligence: Observable-Only Cognitive Impedance Matching for No-Meta LLM-Integrated Systems,2020,3066
390,Behavioral Safety and Context Retention of Large Language Models in a Longitudinal ICU Simulation under Offline Conditions,2023,2639
379,"A Survey on Hallucination in Large Language Models: Principles, Taxonomy, Challenges, and Open Questions",2024,2111
399,PaLM: Scaling Language Modeling with Pathways,2022,2096
470,"Natural language processing: state of the art, current trends and challenges",2022,1820
414,A survey on large language model based autonomous agents,2024,1649


In [3]:
rag_mask = (
    df["title"]
    .fillna("")
    .str.lower()
    .str.contains(
        r"\brag\b|retrieval.?augmented",
        regex=True
    )
)

print(
    "Titles explicitly mentioning RAG:",
    rag_mask.sum()
)

Titles explicitly mentioning RAG: 765


In [4]:
df.loc[
    rag_mask,
    ["title", "year", "citation_count"]
].sort_values(
    "citation_count",
    ascending=False
).head(50)

,title,year,citation_count
96,A Survey on RAG Meeting LLMs: Towards Retrieval-Augmented Large Language Models,2024,757
0,Retrieval-Augmented Generation for Large Language Models: A Survey,2023,748
367,REALM: Retrieval-Augmented Language Model Pre-Training,2020,513
2,RAGAs: Automated Evaluation of Retrieval Augmented Generation,2024,487
1,Active Retrieval Augmented Generation,2023,454
368,In-Context Retrieval-Augmented Language Models,2023,407
256,Almanac — Retrieval-Augmented Language Models for Clinical Medicine,2024,401
3,Benchmarking Large Language Models in Retrieval-Augmented Generation,2024,382
6,Improving the Domain Adaptation of Retrieval Augmented Generation (RAG) Models for Open Domain Question Answering,2023,316
4,Benchmarking Retrieval-Augmented Generation for Medicine,2024,269


In [14]:
import re


def classify_paper(title):
    title = str(title).lower()

    # --------------------------------------------------
    # 1. GRAPH RAG
    # --------------------------------------------------
    if (
        "graph rag" in title
        or "graph retrieval-augmented" in title
        or "knowledge graph" in title and "rag" in title
    ):
        return "GRAPH_RAG"

    # --------------------------------------------------
    # 2. MULTIMODAL RAG
    # --------------------------------------------------
    if (
        "multimodal rag" in title
        or "multimodal retrieval-augmented" in title
        or "retrieval-augmented" in title
        and "image" in title
    ):
        return "MULTIMODAL"

    # --------------------------------------------------
    # 3. AGENTIC / ADAPTIVE RAG
    # --------------------------------------------------
    if any(
    term in title
    for term in [
        "agentic rag",
        "agent rag",
        "adaptive-rag",
        "adaptive rag",
        "corrective retrieval augmented",
        "self-rag",
        "active retrieval augmented",
                ]
    ):
        return "AGENTIC"

    # --------------------------------------------------
    # 4. RAG EVALUATION
    # --------------------------------------------------
    if (
        "ragas" in title
        or (
            ("rag" in title or "retrieval-augmented" in title)
            and any(
                term in title
                for term in [
                    "evaluation",
                    "evaluating",
                    "benchmark",
                ]
            )
        )
    ):
        return "EVALUATION"

    # --------------------------------------------------
    # 5. CORE RAG
    # --------------------------------------------------
    if (
        "retrieval-augmented generation" in title
        or "retrieval augmented generation" in title
        or "retrieval-augmented language model" in title
        or "retrieval augmented language model" in title
        or " rag " in f" {title} "
    ):
        return "CORE_RAG"

    # --------------------------------------------------
    # 6. RETRIEVAL
    # --------------------------------------------------
    if any(
        term in title
        for term in [
            "dense retrieval",
            "information retrieval",
            "passage retrieval",
            "document retrieval",
            "neural retrieval",
            "retrieval model",
            "retrieval system",
            "retriever",
        ]
    ):
        return "RETRIEVAL"

    # --------------------------------------------------
    # 7. LLM
    # --------------------------------------------------
    if any(
        term in title
        for term in [
            "large language model",
            "large language models",
            "llm",
            "language model",
        ]
    ):
        return "LLM"

    # --------------------------------------------------
    # 8. BACKGROUND
    # --------------------------------------------------
    return "BACKGROUND"

In [15]:
df["paper_category"] = df["title"].apply(
    classify_paper
)

In [16]:
df["paper_category"].value_counts()

paper_category
CORE_RAG      401
LLM           200
GRAPH_RAG      97
BACKGROUND     95
AGENTIC        52
MULTIMODAL     52
EVALUATION     38
RETRIEVAL       9
Name: count, dtype: int64

In [17]:
for category in df["paper_category"].unique():

    print("\n" + "=" * 60)
    print(category)
    print("=" * 60)

    print(
        df.loc[
            df["paper_category"] == category,
            "title"
        ].head(15).to_string(index=False)
    )


CORE_RAG
                                                                                    Retrieval-Augmented Generation for Large Language Models: A Survey
                                     Improving the Domain Adaptation of Retrieval Augmented Generation (RAG) Models for Open Domain Question Answering
                                                           Retrieval augmented generation for large language models in healthcare: A systematic review
                                                                                              LightRAG: Simple and Fast Retrieval-Augmented Generation
                                                                     Interactive AI With Retrieval-Augmented Generation for Next Generation Networking
                                                          Retrieval-Augmented Generation with Knowledge Graphs for Customer Service Question Answering
                                                                  Retrieval-augmente

In [18]:
print("Total papers:", len(df))

print(
    "Papers with abstracts:",
    df["abstract"].notna().sum()
)

print(
    "Papers without abstracts:",
    df["abstract"].isna().sum()
)

print(
    "Abstract coverage:",
    round(
        df["abstract"].notna().mean() * 100,
        2
    ),
    "%"
)

Total papers: 944
Papers with abstracts: 825
Papers without abstracts: 119
Abstract coverage: 87.39 %


In [19]:
df["abstract_length"] = (
    df["abstract"]
    .fillna("")
    .str.split()
    .str.len()
)

df["abstract_length"].describe()

count    944.000000
mean     156.686441
std       87.371218
min        0.000000
25%      118.750000
50%      173.000000
75%      210.250000
max      624.000000
Name: abstract_length, dtype: float64

In [20]:
title_duplicates = (
    df[
        df["title"].notna()
    ]
    .assign(
        title_normalized=lambda x:
        x["title"]
        .str.lower()
        .str.replace(
            r"[^a-z0-9]+",
            " ",
            regex=True
        )
        .str.strip()
    )
)

duplicates = title_duplicates[
    title_duplicates["title_normalized"].duplicated(
        keep=False
    )
].sort_values("title_normalized")

duplicates[
    [
        "title",
        "paper_id",
        "doi",
        "year",
        "citation_count"
    ]
].head(50)

,title,paper_id,doi,year,citation_count
715,A Survey of Personalization: From RAG to Agent,https://openalex.org/W4415160040,https://doi.org/10.48550/arxiv.2504.10147,2025,2
744,A Survey of Personalization: From RAG to Agent,https://openalex.org/W7147234952,https://doi.org/10.1145/3802586,2026,1
280,"A Survey on Hallucination in Large Language Models: Principles, Taxonomy, Challenges, and Open Questions",https://openalex.org/W4388585881,https://doi.org/10.48550/arxiv.2311.05232,2023,218
379,"A Survey on Hallucination in Large Language Models: Principles, Taxonomy, Challenges, and Open Questions",https://openalex.org/W4404534210,https://doi.org/10.1145/3703155,2024,2111
96,A Survey on RAG Meeting LLMs: Towards Retrieval-Augmented Large Language Models,https://openalex.org/W4401857375,https://doi.org/10.1145/3637528.3671470,2024,757
216,A Survey on RAG Meeting LLMs: Towards Retrieval-Augmented Large Language Models,https://openalex.org/W4396882099,https://doi.org/10.48550/arxiv.2405.06211,2024,15
1,Active Retrieval Augmented Generation,https://openalex.org/W4389519118,https://doi.org/10.18653/v1/2023.emnlp-main.495,2023,454
102,Active Retrieval Augmented Generation,https://openalex.org/W4376312546,https://doi.org/10.48550/arxiv.2305.06983,2023,18
200,Adaptive-RAG: Learning to Adapt Retrieval-Augmented Large Language Models through Question Complexity,https://openalex.org/W4401042753,https://doi.org/10.18653/v1/2024.naacl-long.389,2024,218
248,Adaptive-RAG: Learning to Adapt Retrieval-Augmented Large Language Models through Question Complexity,https://openalex.org/W4393119012,https://doi.org/10.48550/arxiv.2403.14403,2024,11


In [21]:
# Number of unique normalized titles
unique_titles = title_duplicates["title_normalized"].nunique()

# Number of records that belong to duplicated-title groups
duplicate_records = title_duplicates[
    title_duplicates["title_normalized"].duplicated(keep=False)
]

print("Total records:", len(df))
print("Unique normalized titles:", unique_titles)
print("Records in duplicate-title groups:", len(duplicate_records))
print(
    "Potential duplicate records:",
    len(duplicate_records) - duplicate_records["title_normalized"].nunique()
)

Total records: 944
Unique normalized titles: 876
Records in duplicate-title groups: 132
Potential duplicate records: 68


In [22]:
duplicate_groups = (
    title_duplicates[
        title_duplicates["title_normalized"].duplicated(
            keep=False
        )
    ]
    .sort_values(
        ["title_normalized", "citation_count"],
        ascending=[True, False]
    )
)

duplicate_groups[
    [
        "title",
        "doi",
        "year",
        "citation_count"
    ]
].head(50)

,title,doi,year,citation_count
715,A Survey of Personalization: From RAG to Agent,https://doi.org/10.48550/arxiv.2504.10147,2025,2
744,A Survey of Personalization: From RAG to Agent,https://doi.org/10.1145/3802586,2026,1
379,"A Survey on Hallucination in Large Language Models: Principles, Taxonomy, Challenges, and Open Questions",https://doi.org/10.1145/3703155,2024,2111
280,"A Survey on Hallucination in Large Language Models: Principles, Taxonomy, Challenges, and Open Questions",https://doi.org/10.48550/arxiv.2311.05232,2023,218
96,A Survey on RAG Meeting LLMs: Towards Retrieval-Augmented Large Language Models,https://doi.org/10.1145/3637528.3671470,2024,757
216,A Survey on RAG Meeting LLMs: Towards Retrieval-Augmented Large Language Models,https://doi.org/10.48550/arxiv.2405.06211,2024,15
1,Active Retrieval Augmented Generation,https://doi.org/10.18653/v1/2023.emnlp-main.495,2023,454
102,Active Retrieval Augmented Generation,https://doi.org/10.48550/arxiv.2305.06983,2023,18
200,Adaptive-RAG: Learning to Adapt Retrieval-Augmented Large Language Models through Question Complexity,https://doi.org/10.18653/v1/2024.naacl-long.389,2024,218
248,Adaptive-RAG: Learning to Adapt Retrieval-Augmented Large Language Models through Question Complexity,https://doi.org/10.48550/arxiv.2403.14403,2024,11


In [24]:
# Rank records within each normalized-title group
# Higher citation count = preferred canonical record

df_clean = (
    title_duplicates
    .sort_values(
        ["title_normalized", "citation_count"],
        ascending=[True, False]
    )
    .drop_duplicates(
        subset="title_normalized",
        keep="first"
    )
    .copy()
)

print("Original records:", len(df))
print("Clean records:", len(df_clean))
print("Records removed:", len(df) - len(df_clean))

Original records: 944
Clean records: 876
Records removed: 68


In [25]:
removed_records = df.loc[
    ~df["paper_id"].isin(df_clean["paper_id"])
].copy()

removed_records[
    [
        "title",
        "paper_id",
        "doi",
        "year",
        "citation_count"
    ]
].sort_values(
    "title"
).head(68)

,title,paper_id,doi,year,citation_count
744,A Survey of Personalization: From RAG to Agent,https://openalex.org/W7147234952,https://doi.org/10.1145/3802586,2026,1
280,"A Survey on Hallucination in Large Language Models: Principles, Taxonomy, Challenges, and Open Questions",https://openalex.org/W4388585881,https://doi.org/10.48550/arxiv.2311.05232,2023,218
216,A Survey on RAG Meeting LLMs: Towards Retrieval-Augmented Large Language Models,https://openalex.org/W4396882099,https://doi.org/10.48550/arxiv.2405.06211,2024,15
681,"AI Agents vs. Agentic AI: A Conceptual Taxonomy, Applications and Challenges",https://openalex.org/W4412505619,https://doi.org/10.70777/si.v2i3.15161,2025,175
547,AMSnet-KG: A Netlist Dataset for LLM-based AMS Circuit Auto-Design Using Knowledge Graph RAG,https://openalex.org/W4404649752,https://doi.org/10.48550/arxiv.2411.13560,2024,1
...,...,...,...,...,...
674,Traditional RAG vs. Agentic RAG: A Comparative Study of Retrieval-Augmented Systems,https://openalex.org/W7140314129,https://doi.org/10.1109/fmlds67896.2025.00073,2025,1
455,Tug-of-War Between Knowledge: Exploring and Resolving Knowledge Conflicts in Retrieval-Augmented Language Models,https://openalex.org/W4392120736,https://doi.org/10.48550/arxiv.2402.14409,2024,3
924,Youtube Multimodal Rag System,https://openalex.org/W7159825397,https://doi.org/10.5281/zenodo.19972012,2026,0
912,workinewgebrie/advanced-multimodal-rag-system: advanced-multimodal-rag-system,https://openalex.org/W7148276744,https://doi.org/10.5281/zenodo.19384371,2026,0


In [30]:
removed_records = title_duplicates.loc[
    ~title_duplicates["paper_id"].isin(df_clean["paper_id"])
].copy()

removed_records.groupby("title_normalized").agg(
    records=("paper_id", "count"),
    years=("year", lambda x: sorted(x.tolist())),
    citations=("citation_count", lambda x: sorted(x.tolist(), reverse=True))
).sort_values("records", ascending=False)

,records,years,citations
title_normalized,,,
almanac retrieval augmented language models for clinical medicine,2,"[2023, 2023]","[43, 28]"
rag and rau a survey on retrieval augmented language model in natural language processing,2,"[2024, 2024]","[8, 4]"
do rag a domain specific qa framework using knowledge graph enhanced retrieval augmented generation,2,"[2025, 2026]","[1, 1]"
kaqg a knowledge graph enhanced rag for difficulty controlled question generation,2,"[2025, 2025]","[2, 1]"
logicscore transferability to multimodal rag systems with textual and visual data,1,[2026],[0]
...,...,...,...
graph retrieval augmented generation for large language models a survey,1,[2024],[22]
hm rag hierarchical multi agent multimodal retrieval augmented generation,1,[2025],[1]
how to build an adaptive ai tutor for any course using knowledge graph enhanced retrieval augmented generation kg rag,1,[2023],[12]


In [33]:
duplicate_groups = title_duplicates[
    title_duplicates["title_normalized"].isin(
        removed_records["title_normalized"]
    )
].sort_values(
    ["title_normalized", "citation_count"],
    ascending=[True, False]
)

duplicate_groups[
    [
        "title",
        "title_normalized",
        "paper_id",
        "doi",
        "year",
        "citation_count"
    ]
]

,title,title_normalized,paper_id,doi,year,citation_count
715,A Survey of Personalization: From RAG to Agent,a survey of personalization from rag to agent,https://openalex.org/W4415160040,https://doi.org/10.48550/arxiv.2504.10147,2025,2
744,A Survey of Personalization: From RAG to Agent,a survey of personalization from rag to agent,https://openalex.org/W7147234952,https://doi.org/10.1145/3802586,2026,1
379,"A Survey on Hallucination in Large Language Models: Principles, Taxonomy, Challenges, and Open Questions",a survey on hallucination in large language models principles taxonomy challenges and open questions,https://openalex.org/W4404534210,https://doi.org/10.1145/3703155,2024,2111
280,"A Survey on Hallucination in Large Language Models: Principles, Taxonomy, Challenges, and Open Questions",a survey on hallucination in large language models principles taxonomy challenges and open questions,https://openalex.org/W4388585881,https://doi.org/10.48550/arxiv.2311.05232,2023,218
96,A Survey on RAG Meeting LLMs: Towards Retrieval-Augmented Large Language Models,a survey on rag meeting llms towards retrieval augmented large language models,https://openalex.org/W4401857375,https://doi.org/10.1145/3637528.3671470,2024,757
...,...,...,...,...,...,...
455,Tug-of-War Between Knowledge: Exploring and Resolving Knowledge Conflicts in Retrieval-Augmented Language Models,tug of war between knowledge exploring and resolving knowledge conflicts in retrieval augmented language models,https://openalex.org/W4392120736,https://doi.org/10.48550/arxiv.2402.14409,2024,3
908,workinewgebrie/advanced-multimodal-rag-system: advanced-multimodal-rag-system,workinewgebrie advanced multimodal rag system advanced multimodal rag system,https://openalex.org/W7148280897,https://doi.org/10.5281/zenodo.19384370,2026,0
912,workinewgebrie/advanced-multimodal-rag-system: advanced-multimodal-rag-system,workinewgebrie advanced multimodal rag system advanced multimodal rag system,https://openalex.org/W7148276744,https://doi.org/10.5281/zenodo.19384371,2026,0
923,Youtube Multimodal Rag System,youtube multimodal rag system,https://openalex.org/W7159804412,https://doi.org/10.5281/zenodo.19972011,2026,0


In [34]:
df_clean = (
    title_duplicates
    .sort_values(
        ["title_normalized", "citation_count", "year", "paper_id"],
        ascending=[True, False, False, True]
    )
    .drop_duplicates(
        subset="title_normalized",
        keep="first"
    )
    .copy()
)

print("Original records:", len(df))
print("Clean records:", len(df_clean))
print("Records removed:", len(df) - len(df_clean))

Original records: 944
Clean records: 876
Records removed: 68


In [35]:
print(df_clean["title_normalized"].duplicated().sum())

0


In [36]:
df_clean.to_parquet(
    "../data/processed/papers_clean.parquet",
    index=False
)

print("Clean dataset saved successfully.")
print("Shape:", df_clean.shape)

Clean dataset saved successfully.
Shape: (876, 18)
